# 03 — Genomic Prediction: Classical Quantitative-Genetics Baseline

**Case Study A — Plant Intelligence Lab**

This notebook fits the first predictive model in the repository: a GBLUP baseline evaluated on genotype-aware holdouts.

The model is

$$
\mathbf y=\mathbf X\boldsymbol\beta+\mathbf u+\boldsymbol\varepsilon,
$$

with

$$
\mathbf u\sim\mathcal N(\mathbf 0,\sigma_g^2\mathbf K),\qquad\boldsymbol\varepsilon\sim\mathcal N(\mathbf 0,\sigma_e^2\mathbf I),
$$

where $\mathbf K$ is the genomic relationship matrix created in Notebook 02. The objective is to measure genomic signal that survives prediction on held-out genotypes, not to maximize an in-sample score.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path('..').resolve()
INTERIM = ROOT/'data'/'interim'/'case_study_a'
PROCESSED = ROOT/'data'/'processed'/'case_study_a'
RESULTS = ROOT/'reports'/'results'
FIGURES = ROOT/'reports'/'figures'
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)


## 1. Load the real modelling artifacts

Run Notebooks 01 and 02 first. This notebook stops if the accession-level phenotypes, genomic relationship matrix, matched accessions, or genotype-aware folds have not been materialized.


In [ ]:
def first_existing(paths):
    return next((p for p in paths if p.exists()), None)

phenotype_path = first_existing([INTERIM/'accession_summary.csv', INTERIM/'accession_phenotype_summary.csv', PROCESSED/'accession_summary.csv'])
k_path = first_existing([PROCESSED/'genomic_relationship_matrix.npy', PROCESSED/'K.npy', INTERIM/'genomic_relationship_matrix.npy'])
accession_path = first_existing([PROCESSED/'model_accessions.csv', PROCESSED/'matched_accessions.csv', INTERIM/'model_accessions.csv'])
fold_path = first_existing([PROCESSED/'genotype_aware_folds.csv', PROCESSED/'validation_groups.csv', INTERIM/'genotype_aware_folds.csv'])

required = {'phenotype':phenotype_path,'K':k_path,'accessions':accession_path,'folds':fold_path}
missing = [k for k,v in required.items() if v is None]
if missing:
    raise FileNotFoundError('Missing real inputs: '+', '.join(missing)+'. Run notebooks 01 and 02 first.')

phenotype = pd.read_csv(phenotype_path)
accessions_df = pd.read_csv(accession_path)
folds = pd.read_csv(fold_path)
K = np.load(k_path)
print(required)


In [ ]:
def acc_col(df):
    for c in df.columns:
        if c.lower() in {'accession_id','accession','id'}: return c
    raise KeyError('No accession identifier column found.')

ap, aa, af = acc_col(phenotype), acc_col(accessions_df), acc_col(folds)
phenotype[ap] = phenotype[ap].astype(str)
accessions_df[aa] = accessions_df[aa].astype(str)
folds[af] = folds[af].astype(str)
model_accessions = accessions_df[aa].tolist()

if 'fold' not in folds.columns:
    c = next((c for c in folds.columns if 'fold' in c.lower() or 'cluster' in c.lower()), None)
    if c is None: raise KeyError('No fold/cluster column found.')
    folds = folds.rename(columns={c:'fold'})
fold_map = folds.set_index(af)['fold'].to_dict()

if K.shape != (len(model_accessions), len(model_accessions)):
    raise ValueError(f'K shape {K.shape} does not match {len(model_accessions)} accessions.')
if any(a not in fold_map for a in model_accessions):
    raise ValueError('Some model accessions do not have genotype-aware fold assignments.')

print('n accessions:', len(model_accessions))
print('n validation groups:', pd.Series([fold_map[a] for a in model_accessions]).nunique())


## 2. Build accession-level phenotype targets

Each regeneration endpoint is evaluated separately. Accession means provide the transparent first benchmark; replicate-aware sensitivity analysis can be added later without replacing this baseline.


In [ ]:
if {'phenotype_name','phenotype_mean'}.issubset(phenotype.columns):
    target_wide = phenotype.pivot_table(index=ap, columns='phenotype_name', values='phenotype_mean', aggfunc='mean')
else:
    numeric = phenotype.select_dtypes(include=[np.number]).columns.tolist()
    target_wide = phenotype.set_index(ap)[numeric]
target_wide.index = target_wide.index.astype(str)
target_wide = target_wide.reindex(model_accessions)
coverage = pd.DataFrame({'target':target_wide.columns,'n_nonmissing':[target_wide[c].notna().sum() for c in target_wide.columns]})
coverage['coverage_rate'] = coverage['n_nonmissing']/len(model_accessions)
coverage


## 3. Estimate variance components by REML

For a training set,

$$\mathbf V=\sigma_g^2\mathbf K+\sigma_e^2\mathbf I,$$

and the genomic shrinkage ratio is

$$\lambda=\frac{\sigma_e^2}{\sigma_g^2}.$$

Variance parameters are estimated inside each training fold only.


In [ ]:
def reml_fit(y, K, jitter=1e-8):
    y = np.asarray(y,float).reshape(-1,1); n=len(y); X=np.ones((n,1))
    def obj(theta):
        sg2,se2=np.exp(theta); V=sg2*K+se2*np.eye(n)+jitter*np.eye(n)
        try:
            cf=cho_factor(V,lower=True,check_finite=False)
            Viy=cho_solve(cf,y,check_finite=False); ViX=cho_solve(cf,X,check_finite=False)
            XtViX=X.T@ViX; beta=np.linalg.solve(XtViX,X.T@Viy); r=y-X@beta
            Vir=cho_solve(cf,r,check_finite=False)
            logdetV=2*np.sum(np.log(np.diag(cf[0]))); sign,logdetX=np.linalg.slogdet(XtViX)
            if sign<=0:return np.inf
            return float(.5*(logdetV+logdetX+float(r.T@Vir)+(n-1)*np.log(2*np.pi)))
        except np.linalg.LinAlgError:return np.inf
    v=max(float(np.var(y,ddof=1)),1e-6)
    opt=minimize(obj,np.log([v/2,v/2]),method='L-BFGS-B')
    if not opt.success: raise RuntimeError(opt.message)
    sg2,se2=np.exp(opt.x); V=sg2*K+se2*np.eye(n)+jitter*np.eye(n)
    cf=cho_factor(V,lower=True,check_finite=False); Viy=cho_solve(cf,y); ViX=cho_solve(cf,X)
    beta=float(np.linalg.solve(X.T@ViX,X.T@Viy)[0,0])
    return {'sigma_g2':float(sg2),'sigma_e2':float(se2),'lambda':float(se2/sg2),'h2':float(sg2/(sg2+se2)),'intercept':beta}

def gblup_predict(y_train,K_train,K_test_train):
    fit=reml_fit(y_train,K_train); mu=fit['intercept']; lam=fit['lambda']
    A=K_train+lam*np.eye(len(y_train))+1e-8*np.eye(len(y_train))
    alpha=cho_solve(cho_factor(A,lower=True),np.asarray(y_train)-mu)
    return mu+K_test_train@alpha, fit


## 4. Genotype-aware cross-validation

Prediction for held-out accessions uses

$$
\widehat{\mathbf y}_S=\widehat\mu+\mathbf K_{ST}(\mathbf K_{TT}+\widehat\lambda\mathbf I)^{-1}(\mathbf y_T-\widehat\mu).
$$

The outer folds come directly from the genomic structure established in Notebook 02.


In [ ]:
def corr(y,p):
    return np.nan if len(y)<3 or np.std(y)==0 or np.std(p)==0 else float(np.corrcoef(y,p)[0,1])

fold_vector=np.array([fold_map[a] for a in model_accessions])
rows=[]; preds=[]
for target in target_wide.columns:
    y=target_wide[target].to_numpy(float); available=np.isfinite(y)
    if available.sum()<30:
        print('Skipping',target,'n=',available.sum()); continue
    for fold in pd.unique(fold_vector[available]):
        te=np.where(available & (fold_vector==fold))[0]; tr=np.where(available & (fold_vector!=fold))[0]
        if len(te)==0 or len(tr)<20: continue
        pred,fit=gblup_predict(y[tr],K[np.ix_(tr,tr)],K[np.ix_(te,tr)])
        rows.append({'target':target,'fold':fold,'n_train':len(tr),'n_test':len(te),'rmse':mean_squared_error(y[te],pred)**.5,'mae':mean_absolute_error(y[te],pred),'r2':r2_score(y[te],pred) if len(te)>1 else np.nan,'rho':corr(y[te],pred),'sigma_g2':fit['sigma_g2'],'sigma_e2':fit['sigma_e2'],'lambda':fit['lambda'],'h2':fit['h2']})
        preds += [{'target':target,'fold':fold,'accession_id':model_accessions[i],'observed':y[i],'predicted':float(p)} for i,p in zip(te,pred)]

cv_results=pd.DataFrame(rows); predictions=pd.DataFrame(preds)
if cv_results.empty: raise RuntimeError('No valid GBLUP folds were produced.')
summary=cv_results.groupby('target',as_index=False).agg(n_folds=('fold','nunique'),mean_rmse=('rmse','mean'),sd_rmse=('rmse','std'),mean_mae=('mae','mean'),mean_r2=('r2','mean'),mean_rho=('rho','mean'),mean_h2=('h2','mean'))
summary


## 5. Persist the first model benchmark

These outputs become the reference point for every later high-dimensional machine-learning model. Later models must use the same modelling population and genotype-aware folds unless a change is explicitly justified.


In [ ]:
cv_results.to_csv(RESULTS/'case_study_a_gblup_fold_metrics.csv',index=False)
summary.to_csv(RESULTS/'case_study_a_gblup_summary.csv',index=False)
predictions.to_csv(RESULTS/'case_study_a_gblup_predictions.csv',index=False)
meta={'model':'GBLUP','validation':'genotype-aware folds from notebook 02','variance_estimation':'REML within training folds','n_model_accessions':len(model_accessions)}
(RESULTS/'case_study_a_gblup_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
print(summary.to_string(index=False))


## Interpretation

This benchmark answers three questions before modern ML is allowed to claim value: Is there reproducible genomic signal? Does it survive unseen-genotype validation? How much unexplained variation remains?

If GBLUP is strong, later ML must demonstrate incremental value. If it is weak, the next models must show whether nonlinear marker structure, treatment interactions, or early biological observations recover useful predictive signal.
